In [ ]:
from google.colab import drive
drive.mount('/content/drive')

drive_path = ('/content/drive/MyDrive/FAKE-1-XYZ-0001-SES.las')

import os
if os.getcwd() != drive_path :
  os.chdir(drive_path)

Mounted at /content/drive


In [ ]:
!pip install rockphypy
!pip install lasio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 2.3 MB/s eta 0:00:00


In [ ]:
import rockphypy as rp
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lasio

In [ ]:
from rockphypy import QI

In [ ]:
# Parameters
Dqz, Kqz, Gqz = 2.65, 36.6, 45 ## grain density, bulk and shear modulus
Dsh, Ksh, Gsh = 2.7, 21, 7 # shale/clay density, bulk and shear modulus
Dc,Kc, Gc =2.65, 36.6, 45 # cement density, bulk and shear modulus
Db, Kb = 1, 2.2 # brine density, bulk modulus
Do, Ko = 0.8, 1.5 # oil density, bulk modulus
Dg, Kg = 0.2, 0.06 # gas density, bulk modulus
phi_c=0.4 # critical porosity
sigma=20 # effective pressure
scheme=2
Cn=8.6
# could be array
vsh=0 # shale volume
#phib_p=[0.3,0.36,0.38,0.39] # define cement porosity for Vp
phib_p=0.3
f= 0.5 # slip factor

In [ ]:
# Read data
las = lasio.read('/content/drive/MyDrive/FAKE-1-XYZ-0001-SES.las')

# Convert the LAS file to a DataFrame
df = las.df()
df.to_csv("/content/drive/MyDrive/FAKE-1-XYZ-0001-SES.las.csv", index=False)

# Calculate estimated total porosity
rho_matrix = 2.65  # matrix density
rho_fluid = 1.135  # fluid density
data['PHI_DENS'] = (rho_matrix - data['RHOB']) / (rho_matrix - rho_fluid)
data['PHIT_ND'] = ((data['PHI_DENS'] + data['NPHI']) / 2)

GR_sand = 34.8
GR_shale = 97.8

# Create the 'VSH_GR' column if it doesn't exist
if 'VSH_GR' not in data.columns:
    # Make sure the 'GR' column exists
    if 'GR' in data.columns:
        data['VSH_GR'] = ((data['GR'] - GR_sand) / (GR_shale - GR_sand)).clip(0, 1)
    else:
        print("Column 'GR' not found to calculate 'VSH_GR'.")

# Define the Vp calculation function

def calculo_vp(dt, unidade_dt):
    if unidade_dt.lower() in ['us/ft', 'µs/ft']:
        return 304800 / dt  # converte para m/s
    elif unidade_dt.lower() in ['us/m', 'µs/m']:
        return 1e6 / dt
    elif unidade_dt.lower() in ['s/m']:
        return 1 / dt
    else:
        print("DT unit not found")

dt_unidade = las.curves['DT'].unit
data['VP'] = calculo_vp(data['DT'], dt_unidade)

# Compute the elastic bounds
phi,vp1,vp2,vp3,vs1,vs2,vs3 = QI.screening(Dqz,Kqz,Gqz,Dsh,Ksh,Gsh,Dc,Kc,Gc,Db,Kb,phib_p,phi_c,sigma,vsh,scheme,f, Cn)

# Create an object with data
qi= QI(data.VP,phi=data.PHIT_ND,Vsh= data.VSH_GR)

# Call the screening plot method
fig=qi.screening_plot(phi,vp1,vp2,vp3)
plt.ylim([1900,6100])
plt.yticks(np.arange(2000,6200,1000),[2,3,4,5,6])
plt.ylabel('Vp (km/s)')
plt.xlim(-0.01,0.51)